# From Pseudonymized Medical Records to Neo4J Cypher with the Wowool SDK

In this notebook we will walk you through extracting cypher output form anonymized medical records.

### Objectives:

1. Create a graph schema using the entity-graph.app
2. Using the cypher.app to convert entity-graph results into cypher script.
3. Define custom `.wow` pattern rules to find patient pseudonymized identifiers (Patient_1,Patient_2) and groups for age groups. 
4. Importing the results into Neo4J to visualize them

We will implement this pipeline using a combination of Wowool’s out-of-the-box entities and custom domain rules.

### Data:

For this project we will used previously anonymized data from the project anonymizer-healthrecord.ipymb


# Setup and Package Installation

In [ ]:
!pip install nlp-wowool-sdk wowool-entity-graph wowool-english

### Set up your API key

Sign in to wowool.com to generate an API key. 

For local development, create a .env file in your project directory:

WOWOOL_SDK_KEY="your-api-key-here"

In [ ]:
!pip install python-dotenv

In [ ]:
from dotenv import load_dotenv

load_dotenv()


In [ ]:

! pip install neo4j

Let's look at a one document to clarify what we are going to do:


In [ ]:
"""
# MEDICAL REPORT - CONFIDENTIAL

Patient ID: #**********
Date: Date_1
Hospital: St. Mary's General Hospital
Department: Internal Medicine

## PATIENT INFORMATION:
Name: #Patient_1
DOB: Date_2 (Age: #MIDDLE-AGED_ADULT)
Gender: Female
Address: Address_1
Phone: PhoneNr_1
Insurance: Blue Cross Blue Shield - Policy #: #**********

## ATTENDING PHYSICIAN:
Dr. #Person_1, MD
Internal Medicine
License #: #IL-*****

## CHIEF COMPLAINT:
Patient presents with persistent fatigue, joint pain, and difficulty sleeping over the past 6 weeks.

## MEDICAL HISTORY:
- Hypertension (diagnosed 2019)
- Type 2 Diabetes Mellitus (diagnosed 2021)
- Osteoarthritis (diagnosed 2023)

## CURRENT MEDICATIONS:
1. Metformin 1000mg - twice daily (diabetes management)
2. Lisinopril 10mg - once daily (blood pressure control)
3. Ibuprofen 400mg - as needed for joint pain
4. Vitamin D3 2000 IU - daily

## EXAMINATION FINDINGS:
- Blood Pressure: 142/88 mmHg (elevated)
- Heart Rate: 78 bpm
- Temperature: 98.6°F
- Weight: 165 lbs
- Notable joint tenderness in knees and wrists
- Mild swelling observed in finger joints

## LABORATORY RESULTS:
- HbA1c: 7.8% (target <7.0%)
- Fasting glucose: 158 mg/dL
- Creatinine: 1.2 mg/dL
- ESR: 45 mm/hr (elevated)
- CRP: 8.2 mg/L (elevated)

## DIAGNOSIS:
1. Uncontrolled Type 2 Diabetes Mellitus
2. Hypertension, poorly controlled
3. Possible inflammatory arthritis - rheumatoid arthritis suspected

## TREATMENT PLAN:
1. Increase Metformin to 1000mg three times daily
2. Add Glipizide 5mg twice daily for diabetes control
3. Increase Lisinopril to 15mg daily
4. Discontinue Ibuprofen due to kidney function concerns
5. Start Prednisone 10mg daily for 2 weeks (anti-inflammatory)
6. Prescribe Omeprazole 20mg daily (stomach protection with Prednisone)

## REPORTED SIDE EFFECTS:
- Patient reports nausea with current Metformin dose
- Occasional dizziness, possibly related to blood pressure medication
- Mild stomach upset with Ibuprofen use

## FOLLOW-UP:
- Return in 4 weeks for medication review
- Laboratory work in 6 weeks (HbA1c, kidney function)
- Rheumatology consultation scheduled for Date_3
- Patient education provided on diabetes management and joint care

## PHYSICIAN NOTES:
Patient is motivated to improve health outcomes. Discussed lifestyle modifications including dietary changes and low-impact exercise. Will monitor closely for medication tolerance and effectiveness.

Dr. #Person_1, MD
Date: Date_1
Signature: [Electronic Signature on File]
"""

In [ ]:

from wowool.sdk import Pipeline


# Create a pipeline with the following steps
# Patient_1
#  ├─ AGE → AgeGroup
#  ├─ HAS_SYMPTOM → fatigue
#  └─ TAKES → amoxicillin
pipeline = Pipeline(
    [
        "english",
        "entity",
        "healthcare",
        {
            "name": "snippet.app",
            "options": 
                {
                    "source": """
                        rule:{ "#Patient(.)+" } = Patient; 
                        rule:{ "Age" ":" {<>} = AgeGroup};
                    """,
                },
        },
        {
            "name": "entity-graph.app",
            "options": {
                  "nodes": {
                     "_Patient": { "name": "Patient", "store": "first_seen"},
                     "_HealthIssue": { "name": "HealthIssue.lemma.lower()", "label": "HealthIssue"},
                     "_AgeGroup": { "name": "AgeGroup.literal", "label": "AgeGroup"},
                },
                "links": [
                    {"from": "HealthIssue", "to": "Drug", "relation":"TREATMENT"},
                    {"from": "_Patient", "relation": "HAS_SYMPTOM", "to": "_HealthIssue"},
                    {"from": "_Patient", "relation": "TAKES", "to": "Drug"},
                    {"from": "_Patient", "relation": "AGE", "to": "_AgeGroup"},
                ]

            },
        },
        {
            "name": "cypher.app",
            "options": {
                "namespace": "MySpace",
                "collection": "MyCollection",
                "counters": ["HealthIssue", "Drug"],
            },
        },
    ]
)



Let's run the pipeline on our anonymized documents (we have a sample of 3)

In [ ]:
from wowool.document import Document
from pathlib import Path

cypher_rows = []
input_path = Path("data").expanduser()

for ip in Document.glob(input_path, "*.txt"):


    # Process the input text using the create pipeline
    doc = pipeline(ip.data)
    cypher_results = doc.results("cypher.app")
    print(cypher_results)
    
    cypher_rows.extend(cypher_results['cypher'])

for line in cypher_rows:
        print(f"{line};")

### Visualize your graph in Neo4J

* Create a free AuraDB account.
* Create a database instance and give it a name. Save the username and password provided when you create the instance.
* In the left-hand menu, click Instances. Find your database instance, click the [...] menu on the right, and select Inspect.
* Copy the Connection URI.

Copy the 3 variables into your .env file like this:
CONNECTION_URI = "neo4j+s://*****.databases.neo4j.io"
USERNAME = "********"
PASSWORD = "********"


In [ ]:
# these results can be sent to a Neo4j database
#for line in cypher_results["cypher"]:
#    print(f"{line};")
from dotenv import load_dotenv
from neo4j import GraphDatabase
import os


load_dotenv()
CONNECTION_URI = os.getenv("CONNECTION_URI")
USERNAME = os.getenv("USERNAME")
PASSWORD = os.getenv("PASSWORD")

driver = GraphDatabase.driver(
    CONNECTION_URI,
    auth=(USERNAME, PASSWORD)
)

with driver.session() as session:
    for line in cypher_results["cypher"]:
        r = session.run(line)
        print(r.data())


driver.close()

